# 주식관련사채의 현황과 공시 이력

SEIBRO의 채권별 현황, DART 발행결정 공시, DART 정기보고서 기재 잔액을 함께 봅니다.
세 표의 시점과 식별자가 다르므로 원문 공시와 함께 확인합니다.

[![Colab에서 열기](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/beaten-by-the-market/seibro-api/blob/main/scenarios/combined/03_bond_summary.ipynb)

## 0. 준비
Colab 왼쪽 🔑(Secrets)에 `SEIBRO_API_KEY`, `DART_API_KEY`를 등록하고 이 노트북의 액세스를 켭니다. SEIBRO Open API 키는 [SEIBRO OpenPlatform](https://seibro.or.kr/OpenPlatform/)에서 신청합니다. 아래 두 셀을 실행해 패키지를 설치하고 실행 환경을 준비하세요.

In [1]:
# Colab에서만: 패키지 설치 (로컬은 이 레포를 pip install -e . 로 설치해 두면 됨)
import sys
if "google.colab" in sys.modules:
    !pip -q install "git+https://github.com/beaten-by-the-market/seibro-api.git" koreanize-matplotlib

In [2]:
SECRETS = ['SEIBRO_API_KEY', 'DART_API_KEY']
# ── SEIBRO 공통 셀 (모든 시나리오 노트북 공통) ─────────────────────────────
# Colab: 왼쪽 🔑(Secrets)에 아래 SECRETS 이름으로 값을 넣고 "노트북 액세스"를 켜세요.
#        SEIBRO_API_KEY = 예탁결제원 SEIBRO Open API 인증키 (https://seibro.or.kr/OpenPlatform)
#        DART_API_KEY = OpenDART 인증키 (DART 데이터를 쓰는 노트북만)
#        KRX_ID / KRX_PW = KRX 정보데이터시스템 계정 (KRX 데이터를 쓰는 노트북만)
# 로컬: 환경변수 또는 작업 폴더(레포 루트)의 .env 를 읽습니다.
# 키·비밀번호 값은 어디에도 출력하지 않습니다. 있는지 여부만 보여 줍니다.
import os, sys, warnings
from datetime import date, datetime, timedelta

import pandas as pd

IN_COLAB = "google.colab" in sys.modules
warnings.filterwarnings("ignore", message="Unverified HTTPS request")

if IN_COLAB and SECRETS:
    from google.colab import userdata
    for _name in SECRETS:  # noqa: F821  (빌더가 이 셀 맨 위에 SECRETS 목록을 넣음)
        if os.environ.get(_name):
            continue
        try:
            _v = userdata.get(_name)
        except Exception:
            _v = None
        if not _v:
            from getpass import getpass
            _v = getpass(f"{_name}: ")
        os.environ[_name] = _v.strip()
elif SECRETS:
    from dotenv import find_dotenv, load_dotenv
    load_dotenv(find_dotenv(usecwd=True))

_missing = [
    n for n in SECRETS  # noqa: F821
    if not os.environ.get(n) and (IN_COLAB or not n.startswith("KRX_"))
]
if _missing:
    raise RuntimeError(f"{_missing} 가 없습니다. Colab Secrets 또는 .env 에 넣어 주세요.")

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 50)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")


def won(v, digits=1):
    """원 → '1.2조' / '5,027억' / '3,400만'."""
    if v is None or pd.isna(v):
        return "-"
    a = abs(v)
    if a >= 1e12:
        return f"{v / 1e12:.{digits}f}조"
    if a >= 1e8:
        return f"{v / 1e8:,.0f}억"
    if a >= 1e4:
        return f"{v / 1e4:,.0f}만"
    return f"{v:,.0f}"


def shares(v):
    """주식수 → '23,607,980주'."""
    return "-" if v is None or pd.isna(v) else f"{int(v):,}주"


# 한글 차트 폰트
import matplotlib.pyplot as plt
try:
    import koreanize_matplotlib  # noqa: F401  (Colab: 위 셀에서 설치)
except ImportError:
    if sys.platform.startswith("win"):
        plt.rcParams["font.family"] = "Malgun Gothic"
    elif sys.platform == "darwin":
        plt.rcParams["font.family"] = "AppleGothic"
plt.rcParams["axes.unicode_minus"] = False

def _status(n):
    if os.environ.get(n):
        return "있음"
    # 로컬에서 krx-data-api를 편집 설치(pip install -e)했다면 그 레포의 .env를 패키지가 직접 읽는다
    return "없음" if IN_COLAB or not n.startswith("KRX_") else "krx-data-api 자체 .env 사용"


credential_status = ", ".join(f"{n}: {_status(n)}" for n in SECRETS)
print("SEIBRO 준비 완료", "(Colab)" if IN_COLAB else "(로컬)",
      "| " + (credential_status or "웹 조회 · 키 필요 없음"))

SEIBRO 준비 완료 (로컬) | SEIBRO_API_KEY: 있음, DART_API_KEY: 있음


In [3]:
from IPython.display import display

from seibro_api import get_bonds_from_report, get_dart_cb_events, get_stock_bonds

STOCK_CODE = "317770"  # 엑스페릭스
YEARS = 3
# 이 시나리오는 SEIBRO_API_KEY와 DART_API_KEY를 사용합니다.

In [4]:
current = get_stock_bonds(STOCK_CODE, save_csv=False)
print(f"SEIBRO 사채 현황 · {len(current)}건")
display(current.filter(items=["채권코드", "채권명", "CB/BW/EB", "발행일", "만기일", "미상환잔액", "전환/행사가"]))


[1/4] 종목 317770의 표준코드 조회 중...


  -> 엑스페릭스 (고객번호: 35167, ISIN: KR7317770006)

[2/4] 주식관련사채(CB/BW/EB) 목록 조회 중...


  -> 7건의 사채 발견
     엑스페릭스 5CB(사모/전환/풋) (CB)
     엑스페릭스 6CB(사모/전환/풋) (CB)
     엑스페릭스7CB(사모/전환/풋) (CB)
     엑스페릭스 1CB(사모/전환/풋) (CB)
     엑스페릭스 4CB(사모/전환/풋) (CB)
     엑스페릭스 2BW(사모/신/풋) (비분리형BW)
     엑스페릭스 3BW(사모/신/풋) (비분리형BW)

[3/4] 사채별 상세정보 조회 중...


  OK KR6317771F86


  OK KR6317771G51


  OK KR6317771G93


  OK KR6317771E20


  OK KR6317771E53


  OK KR6317772E29


  OK KR6317773E28

[4/4] 데이터 통합 및 정리 중...

  엑스페릭스(317770) 주식관련사채 조회 결과
  총 7건

  [1] 엑스페릭스 5CB(사모/전환/풋)
      유형: CB | 전환(CB) | 사모
      기간: 20250806 ~ 20280806
      미상환잔액: 4000000000 | 전환/행사가: 2730 | 전환가능주식수: 1465201

  [2] 엑스페릭스 6CB(사모/전환/풋)
      유형: CB | 전환(CB) | 사모
      기간: 20260515 ~ 20290515
      미상환잔액: 15500000000 | 전환/행사가: 3780 | 전환가능주식수: 4100529

  [3] 엑스페릭스7CB(사모/전환/풋)
      유형: CB | 전환(CB) | 사모
      기간: 20260921 ~ 20290921
      미상환잔액: 11000000000 | 전환/행사가: 2396 | 전환가능주식수: 4590984

  [4] 엑스페릭스 1CB(사모/전환/풋)
      유형: CB | 전환(CB) | 사모
      기간: 20240228 ~ 20270228
      미상환잔액: 0 | 전환/행사가: 2467 | 전환가능주식수: 0

  [5] 엑스페릭스 4CB(사모/전환/풋)
      유형: CB | 전환(CB) | 사모
      기간: 20240523 ~ 20270523
      미상환잔액: 1750000000 | 전환/행사가: 3244 | 전환가능주식수: 539457

  [6] 엑스페릭스 2BW(사모/신/풋)
      유형: 비분리형BW | 신주인수권(BW) | 사모
      기간: 20240228 ~ 20270228
      미상환잔액: 4460000000 | 전환/행사가: 3924 | 전환가능주식수: 1136595

  [7] 엑스페릭스 3BW(사모/신/풋)
      유형: 비분리형BW | 신주인수권(BW) | 사모
      기간: 20240228 ~ 2027

,채권코드,채권명,CB/BW/EB,발행일,만기일,미상환잔액,전환/행사가
0,KR6317771F86,엑스페릭스 5CB(사모/전환/풋),CB,20250806,20280806,4000000000,2730
1,KR6317771G51,엑스페릭스 6CB(사모/전환/풋),CB,20260515,20290515,15500000000,3780
2,KR6317771G93,엑스페릭스7CB(사모/전환/풋),CB,20260921,20290921,11000000000,2396
3,KR6317771E20,엑스페릭스 1CB(사모/전환/풋),CB,20240228,20270228,0,2467
4,KR6317771E53,엑스페릭스 4CB(사모/전환/풋),CB,20240523,20270523,1750000000,3244
5,KR6317772E29,엑스페릭스 2BW(사모/신/풋),비분리형BW,20240228,20270228,4460000000,3924
6,KR6317773E28,엑스페릭스 3BW(사모/신/풋),비분리형BW,20240228,20270228,210000000,3924


In [5]:
events = get_dart_cb_events(STOCK_CODE, years=YEARS, save_csv=False)
if "접수번호" in events:
    events["DART 원문"] = events["접수번호"].map(lambda n: f"https://dart.fss.or.kr/dsaf001/main.do?rcpNo={n}")
print(f"DART 발행결정 공시 · {len(events)}건")
display(events.filter(items=[
    "공시유형", "사채종류_회차", "권면총액(원)", "이사회결의일", "납입일", "사채만기일", "DART 원문",
]))


[DART] 317770 주식관련사채 발행 공시 조회
  기간: 20231126 ~ 20261125 (3년)

  [CB] 전환사채발행 조회 중...


    -> 4건

  [BW] 신주인수권부사채발행 조회 중...


    -> 2건

  [EB] 교환사채발행 조회 중...


{'status': '013', 'message': '조회된 데이타가 없습니다.'}
    해당 없음

  총 6건 조회 완료 (CB/BW/EB 통합)
DART 발행결정 공시 · 6건


,공시유형,사채종류_회차,권면총액(원),이사회결의일,납입일,사채만기일,DART 원문
0,CB,4,"4,000,000,000",2024년 05월 22일,2024년 05월 23일,2027년 05월 23일,https://dart.fss.or.kr/dsaf001/main.do?rcpNo=2...
1,CB,5,"4,000,000,000",2025년 07월 29일,2025년 08월 06일,2028년 08월 06일,https://dart.fss.or.kr/dsaf001/main.do?rcpNo=2...
2,CB,6,"15,500,000,000",2026년 05월 07일,2026년 05월 15일,2029년 05월 15일,https://dart.fss.or.kr/dsaf001/main.do?rcpNo=2...
3,CB,7,"11,000,000,000",2026년 05월 07일,2026년 09월 21일,2029년 09월 21일,https://dart.fss.or.kr/dsaf001/main.do?rcpNo=2...
4,BW,2,"11,600,000,000",2024년 02월 26일,2024년 02월 28일,2027년 02월 28일,https://dart.fss.or.kr/dsaf001/main.do?rcpNo=2...
5,BW,3,"8,000,000,000",2024년 02월 26일,2024년 02월 28일,2027년 02월 28일,https://dart.fss.or.kr/dsaf001/main.do?rcpNo=2...


In [6]:
reports = get_bonds_from_report(STOCK_CODE, save_csv=False)
for kind, frame in reports.items():
    details = frame.dropna(subset=["회차"]) if "회차" in frame else frame
    if "rcept_no_new" in details:
        details = details.assign(
            DART_원문=details["rcept_no_new"].map(lambda n: f"https://dart.fss.or.kr/dsaf001/main.do?rcpNo={n}"),
        ).rename(columns={"DART_원문": "DART 원문"})
    print(f"DART 정기보고서 {kind.upper()} · {len(details)}건")
    if not details.empty:
        display(details.filter(items=[
            "보고서유형", "보고서기간", "회차", "발행일", "만기일", "미상환잔액", "기준일", "단위", "DART 원문",
        ]))


[1/3] 317770 최근 사업/반기보고서 조회 중...
  기간: 20241125 ~ 20261125


  -> 사업보고서: 사업보고서 (2025.12) (20260318)
  -> 반기보고서: 반기보고서 (2026.06) (20260814)

[2/3] rcept_no_new 확보 중...
  반기보고서: 20260814001639


  사업보고서: 20260318001711



[3/3] XML 원문 다운로드 중...


  OK 반기보고서 (20260814001639) - 1,762,808자


  OK 사업보고서 (20260318001711) - 2,321,474자


... (7 outputs left out)

,보고서유형,보고서기간,회차,발행일,만기일,미상환잔액,기준일,단위,DART 원문
0,반기보고서,2026.06,4,2024년 05월 23일,2027년 05월 23일,"1,750,000,000",2026년 06월 30일,"(단위 : 원, 주)",https://dart.fss.or.kr/dsaf001/main.do?rcpNo=2...
1,반기보고서,2026.06,5,2025년 08월 06일,2028년 08월 06일,"4,000,000,000",2026년 06월 30일,"(단위 : 원, 주)",https://dart.fss.or.kr/dsaf001/main.do?rcpNo=2...
2,반기보고서,2026.06,6,2026년 05월 15일,2029년 05월 15일,"15,500,000,000",2026년 06월 30일,"(단위 : 원, 주)",https://dart.fss.or.kr/dsaf001/main.do?rcpNo=2...
4,사업보고서,2025.12,4,2024년 05월 23일,2027년 05월 23일,"1,750,000,000",2025년 12월 31일,"(단위 : 원, 주)",https://dart.fss.or.kr/dsaf001/main.do?rcpNo=2...
5,사업보고서,2025.12,5,2025년 08월 06일,2028년 08월 06일,"4,000,000,000",2025년 12월 31일,"(단위 : 원, 주)",https://dart.fss.or.kr/dsaf001/main.do?rcpNo=2...


DART 정기보고서 BW · 4건


,보고서유형,보고서기간,회차,발행일,만기일,미상환잔액,기준일,단위,DART 원문
0,반기보고서,2026.06,2,2024년 02월 28일,2027년 02월 28일,"4,460,000,000",2026년 06월 30일,"(단위 : 원, 주)",https://dart.fss.or.kr/dsaf001/main.do?rcpNo=2...
1,반기보고서,2026.06,3,2024년 02월 28일,2027년 02월 28일,"210,000,000",2026년 06월 30일,"(단위 : 원, 주)",https://dart.fss.or.kr/dsaf001/main.do?rcpNo=2...
3,사업보고서,2025.12,2,2024년 02월 28일,2027년 02월 28일,"4,820,000,000",2025년 12월 31일,"(단위 : 원, 주)",https://dart.fss.or.kr/dsaf001/main.do?rcpNo=2...
4,사업보고서,2025.12,3,2024년 02월 28일,2027년 02월 28일,"890,000,000",2025년 12월 31일,"(단위 : 원, 주)",https://dart.fss.or.kr/dsaf001/main.do?rcpNo=2...


DART 정기보고서 EB · 0건


같은 채권의 보고서 잔액과 조회 시점 잔액을 원 단위로 비교하려면
[미상환 CB·BW 잔액 대조](02_outstanding_bond_reconciliation.md)를 사용합니다.